In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="06-video-recommendation",
    notebook_name="02_candidate_generation.ipynb"
)

# Candidate Generation Deep Dive

## The Big Idea (For a 12-Year-Old)

Imagine you are at the world's biggest library with **10 billion books**. You need to find the 50 best books for your friend -- but you only have **1 second**. You cannot possibly read every book!

So you use a clever trick: first, you quickly grab **a few thousand books** from the right sections (science fiction, mystery, cooking -- whatever your friend likes). This is **candidate generation** -- the fast, rough first pass.

THEN you carefully look at those few thousand books and pick the best 50. That is ranking (next notebook).

In this notebook, we build the "grabbing a few thousand" part using:
1. **Matrix Factorization** -- decompose a huge user-video table into compact embeddings
2. **Two-Tower Neural Network** -- two separate neural networks that learn user and video representations
3. **ANN (Approximate Nearest Neighbor) Search** -- find similar items blazingly fast

---

## Staff-Level Technical Summary

- **Matrix Factorization**: Feedback matrix decomposition, weighted loss (observed + unobserved), WALS optimization
- **Two-Tower NN**: Dual encoder architecture, cross-entropy training, CF and content-based modes
- **ANN Search**: Tree-based, LSH, clustering-based (IVF), HNSW; FAISS practical demo
- **Multiple candidate generators** in parallel for diversity

## 1. Matrix Factorization

### The Feedback Matrix

Imagine a huge table (matrix) where every **row is a user** and every **column is a video**. Each cell says whether that user found that video relevant. Most cells are EMPTY because a user has only seen a tiny fraction of all videos.

Matrix factorization tries to **fill in the missing cells** by discovering hidden patterns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# === The Feedback Matrix ===

# Simulating a user-video feedback matrix
# 1 = user found video relevant (liked or watched >= 50%)
# 0 = unknown (NOT necessarily negative!)
# -1 = user explicitly disliked

n_users = 6
n_videos = 8
video_names = ['Dog Video', 'Cat Funny', 'Python Tutorial', 'Cooking Show',
               'Gaming', 'Music Video', 'News Clip', 'Travel Vlog']
user_names = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank']

# The sparse feedback matrix (most entries unknown = 0)
feedback_matrix = np.array([
    [1, 1, 0, 0, 0, 0, 0, 1],   # Alice likes animals and travel
    [0, 0, 1, 0, 1, 0, 0, 0],   # Bob likes tech and gaming
    [1, 0, 0, 1, 0, 0, 0, 1],   # Charlie likes animals and cooking/travel
    [0, 0, 1, 0, 1, 0, 1, 0],   # Diana likes tech, gaming, news
    [0, 1, 0, 1, 0, 1, 0, 0],   # Eve likes cats, cooking, music
    [0, 0, 0, 0, 0, 1, 1, 0],   # Frank likes music and news
])

# Visualize the feedback matrix
fig, ax = plt.subplots(figsize=(12, 6))

# Color map: 0=gray (unknown), 1=green (positive)
colors = np.where(feedback_matrix == 1, 0.8, 0.2)
im = ax.imshow(colors, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

# Labels
ax.set_xticks(range(n_videos))
ax.set_xticklabels(video_names, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(n_users))
ax.set_yticklabels(user_names, fontsize=10)

# Add text in cells
for i in range(n_users):
    for j in range(n_videos):
        val = feedback_matrix[i, j]
        text = '1' if val == 1 else '?'
        color = 'white' if val == 1 else 'gray'
        ax.text(j, i, text, ha='center', va='center', fontsize=12,
                fontweight='bold', color=color)

ax.set_title('User-Video Feedback Matrix\n(Green=Relevant, Gray=Unknown)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Videos', fontsize=12)
ax.set_ylabel('Users', fontsize=12)
plt.tight_layout()
plt.show()

# Stats
total_cells = n_users * n_videos
known_cells = np.sum(feedback_matrix != 0)
sparsity = (total_cells - known_cells) / total_cells * 100

print(f"\nMatrix size: {n_users} users x {n_videos} videos = {total_cells} cells")
print(f"Known entries: {known_cells} ({known_cells/total_cells*100:.1f}%)")
print(f"Unknown entries: {total_cells - known_cells} ({sparsity:.1f}% SPARSE)")
print(f"\nIn real YouTube: 100M users x 10B videos = 10^18 cells")
print(f"With <0.001% filled. Incredibly sparse!")

In [ ]:
# === Building the Feedback Matrix: Explicit vs Implicit vs Combined ===

print("=== How to Build the Feedback Matrix ===")
print("\nWe need to decide: what counts as '1' (relevant)?")

feedback_options = {
    "Explicit Feedback Only": {
        "signals": ["Likes", "Shares"],
        "pros": "Most accurate -- users explicitly said they liked it",
        "cons": "Too sparse! Only ~1-2% of users press 'like'",
        "verdict": "Not enough data to learn patterns"
    },
    "Implicit Feedback Only": {
        "signals": ["Clicks", "Watch time", "Impressions"],
        "pros": "Lots of data -- every interaction counts",
        "cons": "Noisy! A click doesn't mean they liked it",
        "verdict": "Too noisy to trust"
    },
    "Combined (OUR CHOICE)": {
        "signals": ["Explicit likes/shares", "Watched >= 50%"],
        "pros": "Best of both: accurate signals + sufficient data",
        "cons": "Need to tune the 50% threshold",
        "verdict": "WINNER -- aligns with our ML objective"
    }
}

for name, details in feedback_options.items():
    print(f"\n  {name}")
    print(f"    Signals: {', '.join(details['signals'])}")
    print(f"    Pros: {details['pros']}")
    print(f"    Cons: {details['cons']}")
    print(f"    => {details['verdict']}")

In [ ]:
# === Matrix Factorization: The Core Idea ===

# The feedback matrix A (n_users x n_videos) is decomposed into:
#   U (n_users x embedding_dim) -- user embeddings
#   V (n_videos x embedding_dim) -- video embeddings
# Such that: A ~ U x V^T

embedding_dim = 3  # Small for visualization

print("=== Matrix Factorization: The Math ===")
print(f"\nFeedback Matrix A: ({n_users} users x {n_videos} videos)")
print(f"Decompose into:")
print(f"  U: ({n_users} users x {embedding_dim} dims)  -- User embeddings")
print(f"  V: ({n_videos} videos x {embedding_dim} dims) -- Video embeddings")
print(f"\nSo that: A ~= U x V^T")
print(f"\n12-year-old version:")
print(f"  Instead of storing a HUGE table ({n_users}x{n_videos}={n_users*n_videos} numbers),")
print(f"  we store TWO SMALL tables ({n_users}x{embedding_dim} + {n_videos}x{embedding_dim} = {n_users*embedding_dim + n_videos*embedding_dim} numbers)")
print(f"  AND we can predict the unknown cells!")
print(f"\nReal YouTube scale:")
print(f"  Instead of 100M x 10B = 10^18 numbers (impossible to store!)")
print(f"  We store 100M x 256 + 10B x 256 = ~2.6 trillion numbers (still big but feasible)")

In [ ]:
# === Loss Function Comparison ===

print("=== Matrix Factorization Loss Functions ===")
print("\nThe loss function determines HOW we learn the embeddings.")
print("We have three options:\n")

loss_options = [
    {
        "name": "Option 1: Sum over OBSERVED pairs only",
        "formula": "L = sum over (i,j) where A[i,j]=1 of (A[i,j] - U[i] . V[j])^2",
        "problem": "Doesn't learn from missing data! All-ones matrix has zero loss.",
        "analogy": "Only studying the questions you already know -- you never learn new ones!",
        "verdict": "BAD"
    },
    {
        "name": "Option 2: Sum over ALL pairs (observed + unobserved)",
        "formula": "L = sum over ALL (i,j) of (A[i,j] - U[i] . V[j])^2",
        "problem": "Unobserved (zeros) dominate! Predictions collapse to zero.",
        "analogy": "99% of your test is about things you've never seen -- you score 0 on everything!",
        "verdict": "BAD"
    },
    {
        "name": "Option 3: WEIGHTED combination (OUR CHOICE)",
        "formula": "L = sum_observed(A[i,j] - U[i].V[j])^2 + W * sum_unobserved(0 - U[i].V[j])^2",
        "problem": "Need to tune hyperparameter W, but works well in practice.",
        "analogy": "Study the questions you know AND practice new ones, but focus more on what you know.",
        "verdict": "GOOD -- W balances observed vs unobserved"
    }
]

for opt in loss_options:
    print(f"  {opt['name']}")
    print(f"    Formula: {opt['formula']}")
    print(f"    Problem: {opt['problem']}")
    print(f"    Analogy: {opt['analogy']}")
    print(f"    Verdict: {opt['verdict']}")
    print()

In [ ]:
# === Implementing WALS (Weighted Alternating Least Squares) ===

def wals_matrix_factorization(A, embedding_dim=3, W=0.01, n_iterations=50, reg=0.01):
    """
    Weighted Alternating Least Squares for matrix factorization.
    
    12-year-old version:
    1. Start with random guesses for user and video embeddings
    2. Freeze the video embeddings, find the best user embeddings
    3. Freeze the user embeddings, find the best video embeddings
    4. Repeat until they are good enough
    
    It's like two teams taking turns to improve:
    'OK video team, stay still. User team, adjust yourselves to fit better.'
    'OK user team, stay still. Video team, your turn to adjust.'
    
    Staff-level detail:
    - Each step is a LEAST SQUARES problem (closed-form solution!)
    - Much faster convergence than SGD
    - Parallelizable: each user/video embedding can be updated independently
    - W controls the weight of unobserved entries (typically 0.01-0.1)
    """
    n_users, n_videos = A.shape
    observed_mask = (A > 0).astype(float)
    
    # Weight matrix: observed entries get weight 1, unobserved get weight W
    weight_matrix = observed_mask + W * (1 - observed_mask)
    
    # Initialize randomly
    np.random.seed(42)
    U = np.random.randn(n_users, embedding_dim) * 0.1
    V = np.random.randn(n_videos, embedding_dim) * 0.1
    
    losses = []
    
    for iteration in range(n_iterations):
        # Step 1: Fix V, optimize U
        for i in range(n_users):
            W_i = np.diag(weight_matrix[i])  # Weight vector for user i
            # Closed-form solution: U[i] = (V^T W_i V + reg*I)^-1 V^T W_i A[i]
            VtWV = V.T @ W_i @ V + reg * np.eye(embedding_dim)
            VtWa = V.T @ W_i @ A[i]
            U[i] = np.linalg.solve(VtWV, VtWa)
        
        # Step 2: Fix U, optimize V
        for j in range(n_videos):
            W_j = np.diag(weight_matrix[:, j])  # Weight vector for video j
            UtWU = U.T @ W_j @ U + reg * np.eye(embedding_dim)
            UtWa = U.T @ W_j @ A[:, j]
            V[j] = np.linalg.solve(UtWU, UtWa)
        
        # Compute loss
        predictions = U @ V.T
        errors = weight_matrix * (A - predictions) ** 2
        loss = np.sum(errors) + reg * (np.sum(U**2) + np.sum(V**2))
        losses.append(loss)
        
        if (iteration + 1) % 10 == 0:
            print(f"  Iteration {iteration+1:3d} | Loss: {loss:.4f}")
    
    return U, V, losses


print("=== Training Matrix Factorization with WALS ===")
print(f"\nOriginal feedback matrix ({n_users} users x {n_videos} videos):")
print(feedback_matrix)
print(f"\nLearning {embedding_dim}-dimensional embeddings...\n")

U, V, losses = wals_matrix_factorization(feedback_matrix, embedding_dim=3, W=0.05, n_iterations=50)

In [ ]:
# === Visualize Training Loss ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(losses, 'b-', linewidth=2)
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Weighted Loss', fontsize=12)
axes[0].set_title('WALS Training Loss', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Predicted vs Actual
predicted = U @ V.T
im = axes[1].imshow(predicted, cmap='RdYlGn', vmin=-0.2, vmax=1.2, aspect='auto')
axes[1].set_xticks(range(n_videos))
axes[1].set_xticklabels(video_names, rotation=45, ha='right', fontsize=9)
axes[1].set_yticks(range(n_users))
axes[1].set_yticklabels(user_names, fontsize=10)
axes[1].set_title('Predicted Relevance Scores\n(filled in the missing cells!)', fontsize=13, fontweight='bold')

# Add text
for i in range(n_users):
    for j in range(n_videos):
        original = feedback_matrix[i, j]
        pred = predicted[i, j]
        if original == 1:
            text = f'{pred:.2f}*'  # Star means it was known
            color = 'white'
        else:
            text = f'{pred:.2f}'   # No star means it was predicted
            color = 'black'
        axes[1].text(j, i, text, ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

print("* = originally known (should be close to 1.0)")
print("No star = PREDICTED (these are the recommendations!)")
print("\nFor example, Alice might like 'Cooking Show' with score", f"{predicted[0, 3]:.2f}")
print("and Bob might like 'News Clip' with score", f"{predicted[1, 6]:.2f}")

In [ ]:
# === Inference: Making Recommendations ===

def recommend_for_user(user_idx, U, V, feedback_matrix, video_names, top_k=3):
    """
    Recommend top-k videos for a user.
    
    12-year-old version:
    Take the user's embedding (a little vector of numbers that represents them),
    dot-product it with every video's embedding, and pick the highest scores.
    But skip videos they've already seen!
    """
    # Compute relevance scores (dot product)
    scores = U[user_idx] @ V.T
    
    # Mask out already-seen videos
    already_seen = feedback_matrix[user_idx] > 0
    scores[already_seen] = -np.inf
    
    # Get top-k
    top_indices = np.argsort(-scores)[:top_k]
    
    return [(video_names[idx], scores[idx]) for idx in top_indices]


print("=== Recommendations via Matrix Factorization ===")
for i, user in enumerate(user_names):
    recs = recommend_for_user(i, U, V, feedback_matrix, video_names, top_k=3)
    already_liked = [video_names[j] for j in range(n_videos) if feedback_matrix[i, j] == 1]
    print(f"\n  {user} (liked: {', '.join(already_liked)})")
    print(f"  Recommendations:")
    for rank, (video, score) in enumerate(recs, 1):
        print(f"    #{rank}: {video} (score: {score:.3f})")

## 2. Two-Tower Neural Network

### Why Two Towers?

Matrix factorization only uses **interaction data** (who watched what). It cannot use:
- User features (age, location, device)
- Video features (title, tags, duration)
- Context (time of day, day of week)

The two-tower model fixes this. Think of it as **two separate expert analyzers**:
- **User Tower**: Studies everything about the user and outputs a "user summary vector"
- **Video Tower**: Studies everything about a video and outputs a "video summary vector"

If the two vectors are **close together** in space, the video is relevant to the user!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# === Two-Tower Neural Network in PyTorch ===

class UserTower(nn.Module):
    """
    User Tower: Takes user features and outputs a user embedding.
    
    12-year-old version:
    This is the 'user expert'. It reads everything about you --
    your age, what you've watched, what you've searched for --
    and creates a single summary vector that captures YOUR taste.
    
    Staff-level detail:
    Input features are:
    - User ID embedding (collaborative signal)
    - Demographics: age bucket, gender, language (embeddings)
    - Context: time of day, device, day of week (embeddings)
    - History: averaged embeddings of watched/liked/searched
    """
    def __init__(self, n_users, user_embed_dim=32, n_ages=10, n_genders=3,
                 n_devices=4, history_dim=32, output_dim=64):
        super().__init__()
        
        # Embedding layers for categorical features
        self.user_embedding = nn.Embedding(n_users, user_embed_dim)
        self.age_embedding = nn.Embedding(n_ages, 8)
        self.gender_embedding = nn.Embedding(n_genders, 4)
        self.device_embedding = nn.Embedding(n_devices, 4)
        
        # MLP to combine all features
        total_input = user_embed_dim + 8 + 4 + 4 + history_dim
        self.mlp = nn.Sequential(
            nn.Linear(total_input, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, output_dim),
            nn.ReLU()
        )
    
    def forward(self, user_id, age_bucket, gender, device, watch_history_emb):
        user_emb = self.user_embedding(user_id)
        age_emb = self.age_embedding(age_bucket)
        gender_emb = self.gender_embedding(gender)
        device_emb = self.device_embedding(device)
        
        # Concatenate all features
        combined = torch.cat([user_emb, age_emb, gender_emb, device_emb,
                             watch_history_emb], dim=-1)
        return self.mlp(combined)


class VideoTower(nn.Module):
    """
    Video Tower: Takes video features and outputs a video embedding.
    
    TWO MODES:
    1. CF mode: Only uses video ID (embedding lookup) -- fast, no video features
    2. Content-based mode: Uses full video features (title, tags, duration)
    """
    def __init__(self, n_videos, video_embed_dim=32, n_languages=20,
                 title_dim=64, tag_dim=32, output_dim=64, cf_mode=True):
        super().__init__()
        self.cf_mode = cf_mode
        
        self.video_embedding = nn.Embedding(n_videos, video_embed_dim)
        
        if not cf_mode:
            # Content-based features
            self.language_embedding = nn.Embedding(n_languages, 8)
            total_input = video_embed_dim + 8 + title_dim + tag_dim + 1  # +1 for duration
            self.mlp = nn.Sequential(
                nn.Linear(total_input, 128),
                nn.ReLU(),
                nn.BatchNorm1d(128),
                nn.Dropout(0.2),
                nn.Linear(128, output_dim),
                nn.ReLU()
            )
        else:
            # CF mode: just a linear projection from embedding to output
            self.projection = nn.Linear(video_embed_dim, output_dim)
    
    def forward(self, video_id, language=None, title_emb=None, tag_emb=None, duration=None):
        video_emb = self.video_embedding(video_id)
        
        if self.cf_mode:
            return self.projection(video_emb)
        else:
            lang_emb = self.language_embedding(language)
            combined = torch.cat([video_emb, lang_emb, title_emb, tag_emb,
                                 duration.unsqueeze(-1)], dim=-1)
            return self.mlp(combined)


class TwoTowerModel(nn.Module):
    """
    Complete Two-Tower model.
    
    The similarity between user and video embeddings determines relevance.
    Trained with cross-entropy loss (binary classification: relevant or not).
    """
    def __init__(self, user_tower, video_tower):
        super().__init__()
        self.user_tower = user_tower
        self.video_tower = video_tower
    
    def forward(self, user_features, video_features):
        user_emb = self.user_tower(*user_features)
        video_emb = self.video_tower(video_features[0])  # CF mode: just video ID
        
        # Cosine similarity (or dot product) between embeddings
        # Normalize for cosine similarity
        user_emb = nn.functional.normalize(user_emb, dim=-1)
        video_emb = nn.functional.normalize(video_emb, dim=-1)
        
        similarity = torch.sum(user_emb * video_emb, dim=-1)  # dot product
        return similarity


# Display architecture
print("=== Two-Tower Neural Network Architecture ===")
print()
print("  USER TOWER                    VIDEO TOWER")
print("  ---------                     -----------")
print("  User ID -> Embed(32)          Video ID -> Embed(32)")
print("  Age -> Embed(8)               [CF mode: just projection]")
print("  Gender -> Embed(4)            [Content mode: + lang, title,")
print("  Device -> Embed(4)                          tags, duration]")
print("  Watch History -> Avg(32)")
print("        |                              |")
print("  Concat(80)                     Concat(varies)")
print("        |                              |")
print("  Linear(128) + ReLU             Linear(128) + ReLU")
print("  BN + Dropout                   BN + Dropout")
print("  Linear(64) + ReLU              Linear(64) + ReLU")
print("        |                              |")
print("  User Embedding (64d)          Video Embedding (64d)")
print("        \\                            /")
print("         \\         COSINE          /")
print("          \\      SIMILARITY       /")
print("           \\        |            /")
print("            -> Relevance Score <-")

In [ ]:
# === Training the Two-Tower Model ===

# Create synthetic training data
np.random.seed(42)
torch.manual_seed(42)

n_users_sim = 1000
n_videos_sim = 5000
n_samples = 20000

# Generate positive and negative pairs
user_ids = torch.randint(0, n_users_sim, (n_samples,))
video_ids = torch.randint(0, n_videos_sim, (n_samples,))
age_buckets = torch.randint(0, 10, (n_samples,))
genders = torch.randint(0, 3, (n_samples,))
devices = torch.randint(0, 4, (n_samples,))
watch_history_embs = torch.randn(n_samples, 32)  # Pre-computed averaged embeddings

# Labels: ~20% positive, ~80% negative (realistic imbalance)
# Positive = user liked or watched >= 50%
labels = (torch.rand(n_samples) < 0.2).float()

# Create model
user_tower = UserTower(n_users_sim, output_dim=64)
video_tower = VideoTower(n_videos_sim, output_dim=64, cf_mode=True)
model = TwoTowerModel(user_tower, video_tower)

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
print("=== Training Two-Tower Model ===")
print(f"Users: {n_users_sim}, Videos: {n_videos_sim}, Samples: {n_samples}")
print(f"Positive ratio: {labels.mean():.1%}")
print(f"Loss: BCEWithLogits (cross-entropy for binary classification)")
print()

batch_size = 256
dataset = TensorDataset(user_ids, age_buckets, genders, devices,
                        watch_history_embs, video_ids, labels)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

train_losses = []
model.train()

for epoch in range(15):
    epoch_loss = 0
    n_batches = 0
    
    for batch in loader:
        uid, age, gen, dev, hist, vid, lbl = batch
        
        optimizer.zero_grad()
        
        user_features = (uid, age, gen, dev, hist)
        video_features = (vid,)
        
        similarity = model(user_features, video_features)
        loss = criterion(similarity, lbl)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 3 == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# === Visualize Training and Embedding Space ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
axes[0].plot(train_losses, 'b-o', linewidth=2, markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BCE Loss', fontsize=12)
axes[0].set_title('Two-Tower Training Loss', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Visualize learned embeddings (2D projection via PCA)
from sklearn.decomposition import PCA

model.eval()
with torch.no_grad():
    # Get video embeddings for first 200 videos
    sample_vids = torch.arange(200)
    vid_embs = video_tower.video_embedding(sample_vids)
    vid_embs_projected = video_tower.projection(vid_embs)
    vid_embs_np = vid_embs_projected.numpy()
    
    # Get user embeddings for first 50 users
    sample_users = torch.arange(50)
    sample_ages = torch.randint(0, 10, (50,))
    sample_genders = torch.randint(0, 3, (50,))
    sample_devices = torch.randint(0, 4, (50,))
    sample_hist = torch.randn(50, 32)
    
    user_embs_np = user_tower(sample_users, sample_ages, sample_genders,
                              sample_devices, sample_hist).numpy()

# PCA to 2D
all_embs = np.vstack([user_embs_np, vid_embs_np])
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_embs)

user_2d = all_2d[:50]
video_2d = all_2d[50:]

axes[1].scatter(video_2d[:, 0], video_2d[:, 1], c='#90caf9', alpha=0.5, s=20, label='Videos')
axes[1].scatter(user_2d[:, 0], user_2d[:, 1], c='#ef5350', marker='^', s=50, label='Users')
axes[1].set_xlabel('PCA Dimension 1', fontsize=12)
axes[1].set_ylabel('PCA Dimension 2', fontsize=12)
axes[1].set_title('Learned Embedding Space (2D PCA Projection)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Users (red triangles) and videos (blue dots) live in the SAME space.")
print("Users close to videos = those videos are relevant to that user!")
print("This is the key insight: we can use NEAREST NEIGHBOR search.")

In [ ]:
# === Matrix Factorization vs Two-Tower: Head-to-Head ===

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

comparison_data = [
    ['Aspect', 'Matrix Factorization', 'Two-Tower NN'],
    ['Training cost', 'BETTER (fewer params)', 'WORSE (more params)'],
    ['Inference speed', 'BETTER (static embeddings)', 'WORSE (compute at query time)'],
    ['Cold-start handling', 'WORSE (can\'t handle new users)', 'BETTER (uses user features)'],
    ['Recommendation quality', 'WORSE (no features)', 'BETTER (rich features)'],
    ['Feature flexibility', 'NONE (only user-video IDs)', 'HIGH (any features)'],
    ['Best use case', 'Candidate generation (speed)', 'Candidate gen + ranking']
]

colors_cmp = []
for i, row in enumerate(comparison_data):
    if i == 0:
        colors_cmp.append(['#1565c0', '#1565c0', '#1565c0'])
    else:
        row_colors = ['#f5f5f5']
        for val in row[1:]:
            if 'BETTER' in val:
                row_colors.append('#d4edda')
            elif 'WORSE' in val:
                row_colors.append('#f8d7da')
            elif 'HIGH' in val or 'ranking' in val:
                row_colors.append('#d4edda')
            elif 'NONE' in val or 'speed' in val:
                row_colors.append('#fff3cd')
            else:
                row_colors.append('#f5f5f5')
        colors_cmp.append(row_colors)

table = ax.table(cellText=comparison_data, cellColours=colors_cmp,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

for j in range(3):
    table[0, j].get_text().set_color('white')
    table[0, j].get_text().set_fontweight('bold')

ax.set_title('Matrix Factorization vs Two-Tower Neural Network',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 3. Approximate Nearest Neighbor (ANN) Search

### The Problem

After training our two-tower model, we have:
- A user embedding (64-dimensional vector)
- 10 BILLION video embeddings (each 64-dimensional)

We need to find the top-1000 most similar videos to the user. **Exact search** means computing 10 billion dot products -- too slow!

### The Solution: Approximate Nearest Neighbor (ANN)

Trade a tiny bit of accuracy for MASSIVE speed gains. Instead of finding the EXACT top-1000, find approximately the top-1000 (might miss a few, but 100x faster).

In [ ]:
# === ANN Search Methods ===

ann_methods = {
    "1. Tree-Based (KD-tree, Ball Tree)": {
        "how": "Split space into regions using a tree. Navigate to the right region.",
        "analogy": "A library organized by sections and shelves. Go to the right section first.",
        "pros": "Simple, exact in low dimensions",
        "cons": "Performance degrades in high dimensions (curse of dimensionality)",
        "complexity": "O(d * log n) but degrades to O(n) in high-d"
    },
    "2. Locality Sensitive Hashing (LSH)": {
        "how": "Hash similar vectors to the same bucket with high probability.",
        "analogy": "Putting similar-looking photos into the same album. Quick to find matches!",
        "pros": "Works well in high dimensions, theoretically principled",
        "cons": "Needs many hash tables for good accuracy, memory-intensive",
        "complexity": "Sub-linear, depends on hash parameters"
    },
    "3. Clustering-Based (IVF - Inverted File Index)": {
        "how": "Cluster vectors, then only search within the nearest clusters.",
        "analogy": "Organize books by genre, then only search within the right genre.",
        "pros": "Fast, practical, used by FAISS",
        "cons": "Quality depends on number of clusters and probes",
        "complexity": "O(n/k * d) where k = number of clusters"
    },
    "4. Graph-Based (HNSW - Hierarchical Navigable Small World)": {
        "how": "Build a graph where similar items are connected. Navigate the graph.",
        "analogy": "A social network where friends-of-friends leads you to similar people.",
        "pros": "Best accuracy/speed trade-off, state of the art",
        "cons": "High memory usage, expensive to build",
        "complexity": "O(d * log n) with excellent constants"
    }
}

print("=== Approximate Nearest Neighbor Search Methods ===")
for method, details in ann_methods.items():
    print(f"\n  {method}")
    print(f"    How:       {details['how']}")
    print(f"    Analogy:   {details['analogy']}")
    print(f"    Pros:      {details['pros']}")
    print(f"    Cons:      {details['cons']}")
    print(f"    Time:      {details['complexity']}")

In [ ]:
# === FAISS Demo: Practical ANN Search ===

# NOTE: If faiss is not installed, we use sklearn's NearestNeighbors as a fallback.
# In production, FAISS is the gold standard for ANN search.

import time

try:
    import faiss
    HAS_FAISS = True
    print("FAISS is available! Using FAISS for ANN search.")
except ImportError:
    HAS_FAISS = False
    from sklearn.neighbors import NearestNeighbors
    print("FAISS not installed. Using sklearn NearestNeighbors as fallback.")
    print("To install FAISS: pip install faiss-cpu")

# Generate a large set of video embeddings
np.random.seed(42)
n_videos_large = 100000  # 100k videos (small for demo; real = 10B)
embedding_dim_ann = 64

# Create random video embeddings (in practice, these come from the video tower)
video_embeddings = np.random.randn(n_videos_large, embedding_dim_ann).astype(np.float32)

# Normalize (for cosine similarity)
norms = np.linalg.norm(video_embeddings, axis=1, keepdims=True)
video_embeddings = video_embeddings / norms

# Create a user query embedding
user_query = np.random.randn(1, embedding_dim_ann).astype(np.float32)
user_query = user_query / np.linalg.norm(user_query)

top_k = 10

print(f"\nDataset: {n_videos_large:,} videos, {embedding_dim_ann}-dimensional embeddings")
print(f"Query: 1 user embedding")
print(f"Task: Find top-{top_k} most similar videos")

# --- Method 1: Exact (brute force) search ---
start = time.time()
similarities = video_embeddings @ user_query.T
exact_top_k = np.argsort(-similarities.flatten())[:top_k]
exact_time = time.time() - start

print(f"\n=== Exact Search ===")
print(f"  Time: {exact_time*1000:.2f} ms")
print(f"  Top-{top_k} video IDs: {exact_top_k.tolist()}")
print(f"  Top-{top_k} scores: {[f'{similarities[i][0]:.4f}' for i in exact_top_k]}")

# --- Method 2: ANN search ---
if HAS_FAISS:
    # Build IVF index (clustering-based ANN)
    n_clusters = 100  # Number of Voronoi cells
    quantizer = faiss.IndexFlatIP(embedding_dim_ann)  # Inner product
    index = faiss.IndexIVFFlat(quantizer, embedding_dim_ann, n_clusters,
                               faiss.METRIC_INNER_PRODUCT)
    index.train(video_embeddings)
    index.add(video_embeddings)
    index.nprobe = 10  # Search 10 clusters (trade-off: more probes = better accuracy, slower)
    
    start = time.time()
    scores, ann_top_k = index.search(user_query, top_k)
    ann_time = time.time() - start
    
    print(f"\n=== ANN Search (FAISS IVF, nprobe=10) ===")
    print(f"  Time: {ann_time*1000:.2f} ms")
    print(f"  Top-{top_k} video IDs: {ann_top_k[0].tolist()}")
    print(f"  Top-{top_k} scores: {[f'{s:.4f}' for s in scores[0]]}")
    
    # Recall: what fraction of exact results did ANN find?
    recall = len(set(exact_top_k) & set(ann_top_k[0])) / top_k
    speedup = exact_time / ann_time if ann_time > 0 else float('inf')
    print(f"\n  Recall@{top_k}: {recall:.0%} (fraction of exact results found)")
    print(f"  Speedup: {speedup:.1f}x faster than exact search")
    
else:
    # Fallback: sklearn approximate search
    nn_model = NearestNeighbors(n_neighbors=top_k, metric='cosine', algorithm='ball_tree')
    nn_model.fit(video_embeddings)
    
    start = time.time()
    distances, indices = nn_model.kneighbors(user_query)
    ann_time = time.time() - start
    
    print(f"\n=== ANN Search (sklearn BallTree) ===")
    print(f"  Time: {ann_time*1000:.2f} ms")
    print(f"  Top-{top_k} video IDs: {indices[0].tolist()}")
    
    recall = len(set(exact_top_k) & set(indices[0])) / top_k
    speedup = exact_time / ann_time if ann_time > 0 else float('inf')
    print(f"\n  Recall@{top_k}: {recall:.0%}")
    print(f"  Speedup: {speedup:.1f}x")

print(f"\n=== Key Insight ===")
print(f"At YouTube scale (10B videos), exact search would take ~1 second.")
print(f"ANN search takes ~1-10 milliseconds. That's the difference between")
print(f"'the page loads instantly' and 'the page takes forever.'")

In [ ]:
# === Visualize: Exact vs ANN Trade-off ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Speed comparison (log scale)
methods = ['Exact\n(brute force)', 'Tree-based\n(KD-tree)', 'LSH', 'IVF\n(FAISS)', 'HNSW']
speeds_ms = [100, 20, 10, 5, 3]  # Simulated for 10M vectors
recalls = [100, 95, 85, 92, 98]  # Typical recall@100

colors_bar = ['#ef5350', '#ffa726', '#ffee58', '#66bb6a', '#42a5f5']

bars = axes[0].bar(methods, speeds_ms, color=colors_bar, edgecolor='black')
axes[0].set_ylabel('Latency (ms)', fontsize=12)
axes[0].set_title('Search Latency (Lower = Better)', fontsize=13, fontweight='bold')
axes[0].set_yscale('log')
for bar, speed in zip(bars, speeds_ms):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                f'{speed}ms', ha='center', fontsize=10, fontweight='bold')

bars2 = axes[1].bar(methods, recalls, color=colors_bar, edgecolor='black')
axes[1].set_ylabel('Recall@100 (%)', fontsize=12)
axes[1].set_title('Search Accuracy (Higher = Better)', fontsize=13, fontweight='bold')
axes[1].set_ylim(70, 105)
for bar, recall in zip(bars2, recalls):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{recall}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key trade-off: ANN methods sacrifice a few percent of recall for 10-50x speed.")
print("HNSW gives the best accuracy/speed trade-off but uses more memory.")
print("IVF (used by FAISS) is the most practical for very large scale.")

## 4. Multiple Candidate Generators

### Why Use Multiple Generators?

Users might be interested in videos for MANY different reasons:
- Similar to what they have watched (CF-based)
- Trending right now (popularity-based)
- Relevant to their location (geo-based)
- Brand new content they have not seen (freshness-based)

Using a SINGLE candidate generator misses these diverse reasons. In practice, companies run **multiple generators in parallel** and merge results.

In [ ]:
# === Multiple Candidate Generators ===

class CandidateGenerator:
    """Base class for candidate generators."""
    def __init__(self, name, n_candidates=200):
        self.name = name
        self.n_candidates = n_candidates
    
    def generate(self, user_id):
        """Return list of (video_id, score) tuples."""
        raise NotImplementedError


class CFCandidateGenerator(CandidateGenerator):
    """Collaborative filtering: 'Similar users liked these videos'"""
    def generate(self, user_id):
        np.random.seed(user_id * 1)
        vids = np.random.randint(0, 10000, self.n_candidates)
        scores = np.random.rand(self.n_candidates) * 0.5 + 0.5  # 0.5-1.0
        return list(zip(vids.tolist(), scores.tolist()))


class TrendingCandidateGenerator(CandidateGenerator):
    """Trending: 'Everyone is watching this right now'"""
    def generate(self, user_id):
        # Same trending videos for all users
        np.random.seed(42)
        vids = np.random.randint(0, 1000, self.n_candidates)  # From smaller pool
        scores = np.random.rand(self.n_candidates) * 0.3 + 0.3  # Lower baseline
        return list(zip(vids.tolist(), scores.tolist()))


class GeoCandidateGenerator(CandidateGenerator):
    """Location-based: 'Popular in your area'"""
    def generate(self, user_id):
        np.random.seed(user_id * 3)
        vids = np.random.randint(0, 5000, self.n_candidates)
        scores = np.random.rand(self.n_candidates) * 0.4 + 0.2
        return list(zip(vids.tolist(), scores.tolist()))


class FreshnessCandidateGenerator(CandidateGenerator):
    """Freshness: 'Brand new videos you haven't seen'"""
    def generate(self, user_id):
        np.random.seed(user_id * 4 + 100)
        vids = np.random.randint(9000, 10000, self.n_candidates)  # Only new videos
        scores = np.random.rand(self.n_candidates) * 0.3 + 0.1
        return list(zip(vids.tolist(), scores.tolist()))


def merge_candidates(generators, user_id):
    """
    Run all generators in parallel and merge results.
    
    12-year-old version:
    Ask 4 different friends for video suggestions, then combine all their lists.
    If multiple friends suggest the same video, that's a strong signal!
    """
    all_candidates = {}  # video_id -> (max_score, source)
    
    for gen in generators:
        candidates = gen.generate(user_id)
        for vid, score in candidates:
            if vid not in all_candidates or score > all_candidates[vid][0]:
                all_candidates[vid] = (score, gen.name)
    
    return all_candidates


# Run the candidate generation pipeline
generators = [
    CFCandidateGenerator("CF (Similar Users)", n_candidates=500),
    TrendingCandidateGenerator("Trending", n_candidates=200),
    GeoCandidateGenerator("Location-Based", n_candidates=200),
    FreshnessCandidateGenerator("Fresh Content", n_candidates=100)
]

user_id = 42
merged = merge_candidates(generators, user_id)

print(f"=== Multiple Candidate Generation for User {user_id} ===")
print(f"\n  Generator breakdown:")
for gen in generators:
    print(f"    {gen.name}: {gen.n_candidates} candidates")

print(f"\n  Total unique candidates after merging: {len(merged)}")

# Source distribution
source_counts = {}
for vid, (score, source) in merged.items():
    source_counts[source] = source_counts.get(source, 0) + 1

print(f"\n  Source distribution (unique videos from each source):")
for source, count in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"    {source}: {count} videos")

# Top 10 overall
top_10 = sorted(merged.items(), key=lambda x: -x[1][0])[:10]
print(f"\n  Top 10 candidates (sent to ranking stage):")
for rank, (vid, (score, source)) in enumerate(top_10, 1):
    print(f"    #{rank}: Video {vid} (score: {score:.3f}, source: {source})")

In [ ]:
# === Visualize the Candidate Generation Architecture ===

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

ax.text(7, 9.5, 'Candidate Generation: Multiple Generators in Parallel',
        ha='center', fontsize=14, fontweight='bold')

# User query
ax.add_patch(mpatches.FancyBboxPatch((5.5, 8), 3, 1, boxstyle='round,pad=0.15',
             facecolor='#ffcdd2', edgecolor='#c62828', linewidth=2))
ax.text(7, 8.5, 'User Query', ha='center', fontsize=12, fontweight='bold')

# Four generators
gen_info = [
    (1, 'CF Generator\n(Two-Tower)', '#e3f2fd', '#1565c0', '500 candidates'),
    (4.3, 'Trending\nGenerator', '#e8f5e9', '#2e7d32', '200 candidates'),
    (7.6, 'Location\nGenerator', '#fff3e0', '#e65100', '200 candidates'),
    (10.9, 'Freshness\nGenerator', '#f3e5f5', '#6a1b9a', '100 candidates'),
]

for x, label, fcolor, ecolor, count in gen_info:
    ax.add_patch(mpatches.FancyBboxPatch((x, 4.5), 2.5, 2, boxstyle='round,pad=0.15',
                 facecolor=fcolor, edgecolor=ecolor, linewidth=2))
    ax.text(x + 1.25, 5.8, label, ha='center', fontsize=9, fontweight='bold', color=ecolor)
    ax.text(x + 1.25, 4.8, count, ha='center', fontsize=8, style='italic')
    # Arrow from user
    ax.annotate('', xy=(x + 1.25, 6.5), xytext=(7, 8),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

# Merge box
ax.add_patch(mpatches.FancyBboxPatch((4.5, 2), 5, 1.5, boxstyle='round,pad=0.15',
             facecolor='#fce4ec', edgecolor='#c62828', linewidth=2))
ax.text(7, 3, 'MERGE + DEDUP', ha='center', fontsize=12, fontweight='bold', color='#c62828')
ax.text(7, 2.4, '~800 unique candidates', ha='center', fontsize=10, style='italic')

# Arrows from generators to merge
for x, _, _, _, _ in gen_info:
    ax.annotate('', xy=(7, 3.5), xytext=(x + 1.25, 4.5),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

# Arrow to ranking
ax.annotate('', xy=(7, 0.8), xytext=(7, 2),
            arrowprops=dict(arrowstyle='->', lw=2, color='#c62828'))
ax.text(7, 0.3, 'To Ranking Stage', ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black'))

plt.tight_layout()
plt.show()

## 5. Cold-Start Handling Strategies

In [ ]:
# === Cold-Start Strategies for Candidate Generation ===

print("=" * 70)
print("COLD-START HANDLING IN CANDIDATE GENERATION")
print("=" * 70)

strategies = [
    {
        "scenario": "New User (no watch history)",
        "mf_solution": "CANNOT handle -- MF requires interaction data",
        "two_tower_solution": "Use demographics (age, country, language) to compute user embedding",
        "fallback": "Show trending/popular videos",
        "key_insight": "Two-tower's ability to use features is its KEY advantage over MF"
    },
    {
        "scenario": "New Video (no one has watched it)",
        "mf_solution": "CANNOT handle -- no column in feedback matrix",
        "two_tower_solution": "Use content features (title, tags, duration) in content-based mode",
        "fallback": "Include in freshness candidate generator",
        "key_insight": "Content-based video tower creates embeddings from features, not IDs"
    },
    {
        "scenario": "New Platform (no data at all)",
        "mf_solution": "Cannot train at all",
        "two_tower_solution": "Use pre-trained embeddings from BERT/CBOW for video content",
        "fallback": "Show editorially curated content",
        "key_insight": "Transfer learning from pre-trained models bootstraps the system"
    }
]

for s in strategies:
    print(f"\n  Scenario: {s['scenario']}")
    print(f"    Matrix Factorization: {s['mf_solution']}")
    print(f"    Two-Tower NN:         {s['two_tower_solution']}")
    print(f"    Fallback:             {s['fallback']}")
    print(f"    KEY INSIGHT:          {s['key_insight']}")

## Key Takeaways

1. **Matrix Factorization** decomposes the feedback matrix into user and video embeddings. Fast training and serving, but no features and no cold-start handling.

2. **Weighted loss function** (observed + W * unobserved) is critical -- observed-only or all-pairs both fail.

3. **WALS** converges faster than SGD and is parallelizable -- the preferred optimization for MF.

4. **Two-Tower NN** uses separate encoder towers for users and videos. Supports rich features and cold-start handling.

5. **ANN search** (FAISS, HNSW) makes retrieval from billions of embeddings feasible in milliseconds.

6. **Multiple candidate generators** in parallel ensure diversity of recommendation reasons.

7. The candidate generation stage prioritizes **recall over precision** -- false positives are OK because the ranking stage will filter them.

---

### Next: Notebook 3 -- Ranking Models (scoring, re-ranking, diversity)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)